In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName('OlistDataa') \
.getOrCreate()

26/02/15 06:13:05 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
!hadoop fs -ls /data/olist_proc/

Found 10 items
drwxr-xr-x   - root hadoop          0 2026-02-14 16:13 /data/olist_proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/customers_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:22 /data/olist_proc/geolocation_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/order_item_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/orders_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/payments_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:37 /data/olist_proc/product_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:19 /data/olist_proc/products_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/reviews_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:22 /data/olist_proc/sellers.parquet


In [3]:
customers_df = spark.read.parquet('/data/olist_proc/customers_df_cleaned.parquet')
order_item_df = spark.read.parquet('/data/olist_proc/order_item_df_cleaned.parquet')
payments_df = spark.read.parquet('/data/olist_proc/payments_df.parquet')
sellers_df = spark.read.parquet('/data/olist_proc/sellers.parquet')
geolocation_df = spark.read.parquet('/data/olist_proc/geolocation_df.parquet')
products_df = spark.read.parquet('/data/olist_proc/products_df_cleaned.parquet')
reviews_df = spark.read.parquet('/data/olist_proc/reviews_df_cleaned.parquet')
orders_df = spark.read.parquet('/data/olist_proc/orders_df_cleaned.parquet')

In [4]:
# Cache Frequently used Data for Better Performance

orders_df.cache()
customers_df.cache()
order_item_df.cache()

DataFrame[order_id: string, order_item_id: int, product_id: string, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double]

In [5]:
orders_items_joined_df = orders_df.join(order_item_df, 'order_id', 'inner')

In [6]:
orders_items_products_df = orders_items_joined_df.join(products_df, 'product_id', 'inner')

In [7]:
orders_items_products_sellers_df = orders_items_products_df.join(sellers_df, 'seller_id', 'inner')

In [8]:
full_orders_df = orders_items_products_sellers_df.join(customers_df, 'customer_id', 'inner')

In [9]:
# Geolocation Data

full_orders_df = full_orders_df.join(geolocation_df,full_orders_df.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, 'left')

In [10]:
full_orders_df = full_orders_df.join(reviews_df, 'order_id', 'left')

In [11]:
full_orders_df = full_orders_df.join(payments_df, 'order_id', 'left')

In [12]:
full_orders_df.cache()

26/02/14 20:01:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, product_size_category: string, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: string, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: int, review_comme

In [13]:
full_orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [14]:
# Total Revenue Per seller 

from pyspark.sql.functions import *

seller_revenue_df = full_orders_df.groupBy('seller_id').agg(sum('price'))

In [15]:
seller_revenue_df.show()

+--------------------+-------------------+
|           seller_id|         sum(price)|
+--------------------+-------------------+
|8e6cc767478edae94...|  1140954.090000085|
|4d600e08ecbe08258...|  436434.2300000007|
|62de60d81c55c29d7...|             8343.0|
|cb5ff1b9715e99589...|            13235.0|
|038b75b729c8a9a04...|            17979.5|
|b86a47b3366e3b542...|            17646.0|
|acadd4d36859671cb...|           261994.0|
|33ab10be054370c25...| 45320.700000000244|
|b76dba6c951ab00dc...|  296564.2599999938|
|33cbbec1e7e1044aa...| 190508.32000000085|
|7a67c85e85bb2ce85...|2.031279489002914E7|
|3d8fa2f5b647373c8...| 457301.34999999276|
|e5c84227854980f1d...|  3221.539999999998|
|ca77545ca4d2dfd14...|  76038.90000000065|
|a435b009cd956ea60...|  18115.87999999997|
|d2374cbcbb3ca4ab1...|  3361814.349997649|
|7357b52d27cbaa90f...|   417092.880000017|
|f1fdf2d1318657575...|  973.0999999999993|
|26d8a1c7c75d51304...| 1037349.4999999797|
|238fac594e170b59c...|           480044.0|
+----------

In [16]:
# Total Orders Per Customer

total_orders_per_customer = (
    full_orders_df
    .groupBy("customer_id")
    .agg(count("order_id").alias("total_orders"))
    .orderBy("total_orders", ascending=False)
)

total_orders_per_customer.show(10)

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|5c87184371002d49e...|        6876|
|d5f2b3f597c7ccafb...|        6706|
|24e7dc2ff8c071263...|        6597|
|7bb57d182bdc11653...|        6258|
|d22f25a9fadfb1abb...|        6072|
|63b964e79dee32a35...|        6072|
+--------------------+------------+
only showing top 10 rows



In [17]:
# Average Review Score Per Seller

avg_review_per_seller = (
    full_orders_df
    .groupBy("seller_id")
    .agg(avg("review_score").alias("avg_review_score"))
    .orderBy("avg_review_score", ascending=False)
)

avg_review_per_seller.show(10)

+--------------------+----------------+
|           seller_id|avg_review_score|
+--------------------+----------------+
|b3e2faf2c1004d674...|             5.0|
|a61cc04793308395a...|             5.0|
|539ed9e1981167751...|             5.0|
|0c7f30ae9b147eca0...|             5.0|
|33ab10be054370c25...|             5.0|
|9d213f303afae4983...|             5.0|
|4aba6a02a788d3ec8...|             5.0|
|2c538755f1ca9540a...|             5.0|
|05a48cc8859962767...|             5.0|
|94d76e96eedd97625...|             5.0|
+--------------------+----------------+
only showing top 10 rows



In [18]:
# Most Sold Products (Top 10)

top_products = (
    full_orders_df
    .groupBy("product_id", "product_category_name")
    .agg(count("order_id").alias("total_sold"))
    .orderBy("total_sold", ascending=False)
    .limit(10)
)

top_products.show()

+--------------------+---------------------+----------+
|          product_id|product_category_name|total_sold|
+--------------------+---------------------+----------+
|aca2eb7d00ea1a7b8...|     moveis_decoracao|     86740|
|422879e10f4668299...|   ferramentas_jardim|     81110|
|99a4788cb24856965...|      cama_mesa_banho|     78775|
|389d119b48cf3043d...|   ferramentas_jardim|     60248|
|d1c427060a0f73f6b...| informatica_acess...|     59274|
|368c6c730842d7801...|   ferramentas_jardim|     58358|
|53759a2ecddad2bb8...|   ferramentas_jardim|     52654|
|53b36df67ebb7c415...|   relogios_presentes|     52105|
|154e7e31ebfa09220...|         beleza_saude|     42700|
|3dd2a17168ec895c7...| informatica_acess...|     40787|
+--------------------+---------------------+----------+



In [19]:
# Top Customers By Spending

top_customers_spending = (
    full_orders_df
    .groupBy("customer_id")
    .agg(sum("payment_value").alias("total_spent"))
    .orderBy("total_spent", ascending=False)
)

top_customers_spending.show(10)

+--------------------+-------------------+
|         customer_id|        total_spent|
+--------------------+-------------------+
|1ff773612ab8934db...|1.756825199999893E7|
|0c792d32a3251b4f6...|  8254681.600000529|
|78fc46047c4a639e8...|  7488519.999999339|
|1dbc055ccab23ed89...|  7216273.400000708|
|d5f2b3f597c7ccafb...|  6800018.119998923|
|dd3f1762eb601f41c...| 6746388.4800006235|
|10de381f8a8d23fff...|  5184499.500000076|
|30bb84b541c96af98...|  4740404.549999884|
|d72181923840c8895...|  4513322.700000229|
|e7d6802668de6e74d...| 4000041.4000000926|
+--------------------+-------------------+
only showing top 10 rows



In [5]:
# Restart the kernel then run only first 4 cells only 

In [6]:
# if you run all cells then optimized data shows already cache

## Optimized Joins for Data Integration

In [5]:
orders_items_joined_df_op = orders_df.join(order_item_df, 'order_id', 'inner')

In [6]:
orders_items_products_df_op = orders_items_joined_df_op.join(products_df, 'product_id', 'inner')

In [7]:
from pyspark.sql.functions import *

In [8]:
orders_items_products_sellers_df_op = orders_items_products_df_op.join(broadcast(sellers_df), 'seller_id', 'inner')

In [9]:
full_orders_df_op = orders_items_products_sellers_df_op.join(customers_df, 'customer_id', 'inner')

In [10]:
# Geolocation Data
full_orders_df_op = full_orders_df_op.join(broadcast(geolocation_df),full_orders_df_op.customer_zip_code_prefix == geolocation_df.geolocation_zip_code_prefix, 'left')

In [11]:
full_orders_df_op = full_orders_df_op.join(broadcast(reviews_df), 'order_id', 'left')

In [12]:
full_orders_df_op = full_orders_df_op.join(payments_df, 'order_id', 'left')

In [13]:
full_orders_df_op.cache()

26/02/15 06:14:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[order_id: string, customer_id: string, seller_id: string, product_id: string, order_status: string, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, order_item_id: int, shipping_limit_date: timestamp, price: double, freight_value: double, product_category_name: string, product_name_lenght: int, product_description_lenght: int, product_photos_qty: int, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, product_size_category: string, seller_zip_code_prefix: int, seller_city: string, seller_state: string, customer_unique_id: string, customer_zip_code_prefix: string, customer_city: string, customer_state: string, geolocation_zip_code_prefix: int, geolocation_lat: double, geolocation_lng: double, geolocation_city: string, geolocation_state: string, review_id: string, review_score: int, review_comme

In [14]:
# Total Orders Per Customer

total_orders_per_customer = (
    full_orders_df_op
    .groupBy("customer_id")
    .agg(count("order_id").alias("total_orders"))
    .orderBy("total_orders", ascending=False)
)

total_orders_per_customer.show(10)

+--------------------+------------+
|         customer_id|total_orders|
+--------------------+------------+
|351e40989da90e704...|       11427|
|50920f8cd0681fd86...|       10752|
|9b43e2a62de9bab3a...|        8556|
|270c23a11d024a44c...|        8001|
|5c87184371002d49e...|        6876|
|d5f2b3f597c7ccafb...|        6706|
|24e7dc2ff8c071263...|        6597|
|7bb57d182bdc11653...|        6258|
|d22f25a9fadfb1abb...|        6072|
|63b964e79dee32a35...|        6072|
+--------------------+------------+
only showing top 10 rows



In [15]:
# Average Review Score Per Seller

avg_review_per_seller = (
    full_orders_df_op
    .groupBy("seller_id")
    .agg(avg("review_score").alias("avg_review_score"))
    .orderBy("avg_review_score", ascending=False)
)

avg_review_per_seller.show(10)

+--------------------+----------------+
|           seller_id|avg_review_score|
+--------------------+----------------+
|b3e2faf2c1004d674...|             5.0|
|a61cc04793308395a...|             5.0|
|539ed9e1981167751...|             5.0|
|0c7f30ae9b147eca0...|             5.0|
|33ab10be054370c25...|             5.0|
|9d213f303afae4983...|             5.0|
|4aba6a02a788d3ec8...|             5.0|
|2c538755f1ca9540a...|             5.0|
|05a48cc8859962767...|             5.0|
|94d76e96eedd97625...|             5.0|
+--------------------+----------------+
only showing top 10 rows



In [16]:
# Most Sold Products (Top 10)

top_products = (
    full_orders_df_op
    .groupBy("product_id", "product_category_name")
    .agg(count("order_id").alias("total_sold"))
    .orderBy("total_sold", ascending=False)
    .limit(10)
)

top_products.show()

+--------------------+---------------------+----------+
|          product_id|product_category_name|total_sold|
+--------------------+---------------------+----------+
|aca2eb7d00ea1a7b8...|     moveis_decoracao|     86740|
|422879e10f4668299...|   ferramentas_jardim|     81110|
|99a4788cb24856965...|      cama_mesa_banho|     78775|
|389d119b48cf3043d...|   ferramentas_jardim|     60248|
|d1c427060a0f73f6b...| informatica_acess...|     59274|
|368c6c730842d7801...|   ferramentas_jardim|     58358|
|53759a2ecddad2bb8...|   ferramentas_jardim|     52654|
|53b36df67ebb7c415...|   relogios_presentes|     52105|
|154e7e31ebfa09220...|         beleza_saude|     42700|
|3dd2a17168ec895c7...| informatica_acess...|     40787|
+--------------------+---------------------+----------+



In [17]:
# Top Customers By Spending

top_customers_spending = (
    full_orders_df_op
    .groupBy("customer_id")
    .agg(sum("payment_value").alias("total_spent"))
    .orderBy("total_spent", ascending=False)
)

top_customers_spending.show(10)

+--------------------+-------------------+
|         customer_id|        total_spent|
+--------------------+-------------------+
|1ff773612ab8934db...|1.756825199999893E7|
|0c792d32a3251b4f6...|  8254681.600000529|
|78fc46047c4a639e8...|  7488519.999999339|
|1dbc055ccab23ed89...|  7216273.400000708|
|d5f2b3f597c7ccafb...|  6800018.119998923|
|dd3f1762eb601f41c...| 6746388.4800006235|
|10de381f8a8d23fff...|  5184499.500000076|
|30bb84b541c96af98...|  4740404.549999884|
|d72181923840c8895...|  4513322.700000229|
|e7d6802668de6e74d...| 4000041.4000000926|
+--------------------+-------------------+
only showing top 10 rows



## Window Function and Ranking

In [18]:
from pyspark.sql.window import Window

In [19]:
window_spec = Window.partitionBy('seller_id').orderBy(col('price').desc())

In [20]:
# Rank Top Selling products per seller

top_seller_products_df = full_orders_df_op.withColumn('rank',rank().over(window_spec)).filter(col('rank')<=5)

top_seller_products_df.select('seller_id', 'price', 'rank').show(10)

+--------------------+-----+----+
|           seller_id|price|rank|
+--------------------+-----+----+
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
|001cca7ae9ae17fb1...|199.0|   1|
+--------------------+-----+----+
only showing top 10 rows



## Advance Aggregation and Enrichment

In [21]:
# Total Revenue and Average Order Value (AOV) per customer

customer_spending_df = full_orders_df_op.groupBy('customer_id')\
.agg(
    count('order_id').alias('total_orders'),
    sum('price').alias('total_spent'),
    round(avg('price'),2).alias('AOV')
)\
.orderBy(desc('total_spent'))

customer_spending_df.show()

+--------------------+------------+------------------+------+
|         customer_id|total_orders|       total_spent|   AOV|
+--------------------+------------+------------------+------+
|63b964e79dee32a35...|        6072|         2501664.0| 412.0|
|1ff773612ab8934db...|        5820|1658641.7999999512|284.99|
|de832e8dbb1f588a4...|        2190|1584990.5999999817|723.74|
|d72181923840c8895...|        2721|1488114.8999999566| 546.9|
|904805ce22729b9b7...|        3306|         1319094.0| 399.0|
|1dbc055ccab23ed89...|        4535|1314696.4999999965| 289.9|
|9af2372a1e4934027...|        2726|1070091.3000000538|392.55|
|351e40989da90e704...|       11427| 982607.7299999113| 85.99|
|f034ceb6936919a21...|        1102|          935598.0| 849.0|
|8d00bb042dfdfc4c8...|        3960| 934164.0000000617| 235.9|
|5f79f446984a2ba4f...|        3846| 922655.4000000596| 239.9|
|6361b9f3b85d41860...|        1293| 903781.1399999846|698.98|
|5a47f2e6f5067f646...|        1928| 900356.7199999853|466.99|
|dd3f176

In [22]:
2501664/6072

412.0

In [24]:
# Seller Performance Metrics ( Revenue, Average Review, Order Count)

seller_performance_df = full_orders_df_op.groupBy('seller_id') \
    .agg(
        count('order_id').alias('total_orders'),
        sum('price').alias('total_revenue'),
        round(avg('review_score'), 2).alias('avg_review_score'),
        round(stddev('price'), 2).alias('price_variability')
    ) \
    .orderBy(desc('total_revenue'))

seller_performance_df.show()

+--------------------+------------+--------------------+----------------+-----------------+
|           seller_id|total_orders|       total_revenue|avg_review_score|price_variability|
+--------------------+------------+--------------------+----------------+-----------------+
|4869f7a5dfa277a7d...|      184498| 3.605861820999457E7|             4.1|            110.6|
|4a3ca9315b744ce9f...|      330615|   3.3756355440012E7|            3.79|            59.37|
|7c67e1448b00f6e96...|      233304|3.2282087810021453E7|            3.49|            50.39|
|da8622b14eb17ae28...|      264361|2.9856956930036083E7|            4.02|            72.91|
|fa1c13f2614d7b5c4...|       84723|2.6013136650012247E7|            4.39|           204.32|
|1025f0e2d44d7041d...|      229587|2.2937518520012792E7|            3.94|             84.3|
|955fee9216a65b617...|      232364|2.0964410670017645E7|            4.07|            84.94|
|46dc3b2cc0980fb8e...|       89587|2.0583881680015896E7|            4.19|       

In [25]:
# Product Popularity Metrics

product_metrics_df = full_orders_df_op.groupBy('product_id') \
    .agg(
        count('order_id').alias('total_sales'),
        sum('price').alias('total_revenue'),
        round(avg('price'), 2).alias('avg_price'),
        round(stddev('price'), 2).alias('price_volatility'),
        collect_set('seller_id').alias('unique_sellers')
    ) \
    .orderBy(desc('total_sales'))

product_metrics_df.show()

+--------------------+-----------+------------------+---------+----------------+--------------------+
|          product_id|total_sales|     total_revenue|avg_price|price_volatility|      unique_sellers|
+--------------------+-----------+------------------+---------+----------------+--------------------+
|aca2eb7d00ea1a7b8...|      86740| 6164630.299996178|    71.07|            3.17|[955fee9216a65b61...|
|422879e10f4668299...|      81110| 4442791.509997465|    54.77|            4.46|[1f50f920176fa81d...|
|99a4788cb24856965...|      78775| 6921762.709996341|    87.87|            4.08|[4a3ca9315b744ce9...|
|389d119b48cf3043d...|      60248| 3280533.129998829|    54.45|            4.37|[1f50f920176fa81d...|
|d1c427060a0f73f6b...|      59274| 8220103.330002298|   138.68|           16.58|[a1043bafd471dff5...|
|368c6c730842d7801...|      58358|    3181698.899999|    54.52|            4.59|[1f50f920176fa81d...|
|53759a2ecddad2bb8...|      52654| 2893017.499999508|    54.94|            4.52|[1

In [26]:
# collect_set
# Collects all values from a column into an array
# Removes duplicates automatically

In [27]:
# eg
# product_id	seller_id
# P1	S1
# P1	S2
# P1 	S1
# P1	S3

In [28]:
# Output:
# product_id	unique_sellers
# P1	[S1, S2, S3]

In [29]:
# Monthly Revenue & Order Count Trend

monthly_trend_df = full_orders_df_op \
    .withColumn("year", year(col("order_purchase_timestamp"))) \
    .withColumn("month", month(col("order_purchase_timestamp"))) \
    .groupBy("year", "month") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("price"), 2).alias("total_revenue"),
        round(avg("price"), 2).alias("avg_order_value"),
        round(min("price"), 2).alias("min_order_value"),
        round(max("price"), 2).alias("max_order_value")
    ) \
    .orderBy("year", "month")

monthly_trend_df.show()

+----+-----+------------+--------------+---------------+---------------+---------------+
|year|month|total_orders| total_revenue|avg_order_value|min_order_value|max_order_value|
+----+-----+------------+--------------+---------------+---------------+---------------+
|2016|    9|        1268|       57431.0|          45.29|           32.9|           59.5|
|2016|   10|       59808|    7091830.03|         118.58|           10.0|         739.98|
|2016|   12|         304|        3313.6|           10.9|           10.9|           10.9|
|2017|    1|      149515| 1.648761565E7|         110.27|           9.99|          869.0|
|2017|    2|      306973| 3.460486758E7|         112.73|           9.99|          890.0|
|2017|    3|      476829| 4.997020476E7|          104.8|           10.0|          889.0|
|2017|    4|      418426| 4.710977721E7|         112.59|           10.0|          869.9|
|2017|    5|      672514| 7.463973091E7|         110.99|           9.99|          890.0|
|2017|    6|      566

In [31]:
# Customer Retention Analysis (First and Last Order)

customer_retention_df = full_orders_df_op.groupBy('customer_id') \
    .agg(
        min('order_purchase_timestamp').alias('first_order_date'),
        max('order_purchase_timestamp').alias('last_order_date'),
        count('order_id').alias('total_orders'),
        round(avg('price'), 2).alias('aov')
    ) \
    .orderBy(desc('total_orders'))

customer_retention_df.show()

+--------------------+-------------------+-------------------+------------+------+
|         customer_id|   first_order_date|    last_order_date|total_orders|   aov|
+--------------------+-------------------+-------------------+------------+------+
|351e40989da90e704...|2017-07-13 10:42:37|2017-07-13 10:42:37|       11427| 85.99|
|50920f8cd0681fd86...|2018-01-27 11:28:32|2018-01-27 11:28:32|       10752| 43.82|
|9b43e2a62de9bab3a...|2017-05-25 22:27:50|2017-05-25 22:27:50|        8556|  26.4|
|270c23a11d024a44c...|2017-08-08 20:26:31|2017-08-08 20:26:31|        8001| 36.59|
|5c87184371002d49e...|2018-01-05 19:15:37|2018-01-05 19:15:37|        6876| 12.49|
|d5f2b3f597c7ccafb...|2017-12-13 14:21:15|2017-12-13 14:21:15|        6706|  59.0|
|24e7dc2ff8c071263...|2017-11-24 16:16:45|2017-11-24 16:16:45|        6597|  59.2|
|7bb57d182bdc11653...|2018-04-02 17:11:30|2018-04-02 17:11:30|        6258|  86.9|
|d22f25a9fadfb1abb...|2018-05-12 12:28:58|2018-05-12 12:28:58|        6072| 14.99|
|63b

##  Extended Enrichment

In [36]:
# Orders Status Flags

In [39]:
full_orders_df_op.select('order_status').show()

+------------+
|order_status|
+------------+
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
|   delivered|
+------------+
only showing top 20 rows



In [44]:
full_orders_df = (
    full_orders_df_op
    .withColumn(
        "is_delivered",
        when(col("order_status") == "delivered", lit(1)).otherwise(lit(0))
    )
    .withColumn(
        "is_canceled",
        when(col("order_status") == "canceled", lit(1)).otherwise(lit(0))
    )
)

full_orders_df.select("order_status", "is_delivered", "is_canceled").show()

+------------+------------+-----------+
|order_status|is_delivered|is_canceled|
+------------+------------+-----------+
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
|   delivered|           1|          0|
+------------+------------+-----------+
only showing top 20 rows



In [45]:
full_orders_df.where(full_orders_df['order_status'] != 'delivered').select('order_status', 'is_delivered', 'is_canceled').show()

+------------+------------+-----------+
|order_status|is_delivered|is_canceled|
+------------+------------+-----------+
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
|     shipped|           0|          0|
+------------+------------+-----------+
only showing top 20 rows



In [46]:
full_orders_df.where(full_orders_df['order_status'] == 'canceled').select('order_status', 'is_delivered', 'is_canceled').show()

+------------+------------+-----------+
|order_status|is_delivered|is_canceled|
+------------+------------+-----------+
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
|    canceled|           0|          1|
+------------+------------+-----------+
only showing top 20 rows



In [47]:
# Order Revenue Calculation
full_orders_df = full_orders_df.withColumn('order_revenue',col('price') + col('freight_value'))

full_orders_df.select('price', 'freight_value', 'order_revenue').show()

+-----+-------------+-------------+
|price|freight_value|order_revenue|
+-----+-------------+-------------+
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
| 58.9|        13.29|        72.19|
+-----+-------------+-------------+
only showing top 20 rows



In [49]:
customer_spending_df.show()

+--------------------+------------+------------------+------+
|         customer_id|total_orders|       total_spent|   AOV|
+--------------------+------------+------------------+------+
|63b964e79dee32a35...|        6072|         2501664.0| 412.0|
|1ff773612ab8934db...|        5820|1658641.7999999512|284.99|
|de832e8dbb1f588a4...|        2190|1584990.5999999817|723.74|
|d72181923840c8895...|        2721|1488114.8999999566| 546.9|
|904805ce22729b9b7...|        3306|         1319094.0| 399.0|
|1dbc055ccab23ed89...|        4535|1314696.4999999965| 289.9|
|9af2372a1e4934027...|        2726|1070091.3000000538|392.55|
|351e40989da90e704...|       11427| 982607.7299999113| 85.99|
|f034ceb6936919a21...|        1102|          935598.0| 849.0|
|8d00bb042dfdfc4c8...|        3960| 934164.0000000617| 235.9|
|5f79f446984a2ba4f...|        3846| 922655.4000000596| 239.9|
|6361b9f3b85d41860...|        1293| 903781.1399999846|698.98|
|5a47f2e6f5067f646...|        1928| 900356.7199999853|466.99|
|dd3f176

In [50]:
customer_spending_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- total_orders: long (nullable = false)
 |-- total_spent: double (nullable = true)
 |-- AOV: double (nullable = true)



In [52]:
# Customer Segmentation based on spending

customer_spending_df = customer_spending_df.withColumn(
    'customer_segment',
    when(col('AOV') >= 700, "High-Value")
    .when((col('AOV') < 700) & (col('AOV') >= 400), "Medium_Value")
    .otherwise("Low-Value")
)

customer_spending_df.show()

+--------------------+------------+------------------+------+----------------+
|         customer_id|total_orders|       total_spent|   AOV|customer_segment|
+--------------------+------------+------------------+------+----------------+
|63b964e79dee32a35...|        6072|         2501664.0| 412.0|    Medium_Value|
|1ff773612ab8934db...|        5820|1658641.7999999512|284.99|       Low-Value|
|de832e8dbb1f588a4...|        2190|1584990.5999999817|723.74|      High-Value|
|d72181923840c8895...|        2721|1488114.8999999566| 546.9|    Medium_Value|
|904805ce22729b9b7...|        3306|         1319094.0| 399.0|       Low-Value|
|1dbc055ccab23ed89...|        4535|1314696.4999999965| 289.9|       Low-Value|
|9af2372a1e4934027...|        2726|1070091.3000000538|392.55|       Low-Value|
|351e40989da90e704...|       11427| 982607.7299999113| 85.99|       Low-Value|
|f034ceb6936919a21...|        1102|          935598.0| 849.0|      High-Value|
|8d00bb042dfdfc4c8...|        3960| 934164.000000061

In [53]:
full_orders_df = full_orders_df_op.join(customer_spending_df.select('customer_id', 'customer_segment'),'customer_id',how='left')
full_orders_df.select('customer_id', 'customer_segment').show()

+--------------------+----------------+
|         customer_id|customer_segment|
+--------------------+----------------+
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
|c08ef557085ca9fb0...|       Low-Value|
+--------------------+----------------+
only showing top 20 rows



In [56]:
full_orders_df_op.select('order_purchase_timestamp').show()

+------------------------+
|order_purchase_timestamp|
+------------------------+
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
|     2017-09-13 08:59:02|
+------------------------+
only showing top 20 rows



In [57]:
# Hourly Order Distribution

full_orders_df_op = full_orders_df_op.withColumn(
    'hour_of_day',
    expr('hour(order_purchase_timestamp)')
)


full_orders_df_op.select('order_purchase_timestamp', 'hour_of_day').show()

+------------------------+-----------+
|order_purchase_timestamp|hour_of_day|
+------------------------+-----------+
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
|     2017-09-13 08:59:02|          8|
+------------------------+-----------+
only showing top 20 rows



In [58]:
# expr
# xpr() lets you write SQL expressions inside PySpark.

In [59]:
# Weekday VS Weekend Order

full_orders_df_op = full_orders_df_op.withColumn(
    'order_day_type',
    when(dayofweek('order_purchase_timestamp').isin(1, 7), lit('Weekend'))
    .otherwise(lit('Weekday'))
)

full_orders_df_op.select('order_purchase_timestamp', 'order_day_type').show()

+------------------------+--------------+
|order_purchase_timestamp|order_day_type|
+------------------------+--------------+
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
|     2017-09-13 08:59:02|       Weekday|
+------------------------+--------

In [60]:
# In pyspark
# 1 -> Sunday
# 7 -> Saturday

In [61]:
full_orders_df_op.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

In [62]:
# Freight Category Column

full_orders_df_op = full_orders_df_op.withColumn(
    "freight_category",
    when(col("freight_value") < 20, lit("Low"))
    .when((col("freight_value") >= 20) & (col("freight_value") <= 50), lit("Medium"))
    .otherwise(lit("High"))
)

full_orders_df_op.select("freight_value", "freight_category").show()

+-------------+----------------+
|freight_value|freight_category|
+-------------+----------------+
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
|        13.29|             Low|
+-------------+----------------+
only showing top 20 rows



In [63]:
# Order Volume by Customer State

order_volume_state = (
    full_orders_df_op
    .groupBy("customer_state")
    .agg(count("order_id").alias("total_orders"))
    .orderBy(col("total_orders").desc())
)

order_volume_state.show()

+--------------+------------+
|customer_state|total_orders|
+--------------+------------+
|            SP|     6606895|
|            RJ|     3546809|
|            MG|     3364839|
|            RS|      954695|
|            PR|      727091|
|            SC|      632815|
|            BA|      434138|
|            ES|      361954|
|            GO|      158840|
|            MT|      152565|
|            PE|      130118|
|            DF|      107772|
|            PA|       92299|
|            CE|       72889|
|            MS|       71356|
|            MA|       60679|
|            AL|       36214|
|            PB|       32322|
|            SE|       27738|
|            PI|       27313|
+--------------+------------+
only showing top 20 rows



In [64]:
full_orders_df_op.write.mode("overwrite").parquet("/data/olist_proc/full_orders_df_op.parquet")

In [65]:
!hadoop fs -ls /data/olist_proc/

Found 11 items
drwxr-xr-x   - root hadoop          0 2026-02-14 16:13 /data/olist_proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/customers_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-15 07:57 /data/olist_proc/full_orders_df_op.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:22 /data/olist_proc/geolocation_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/order_item_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/orders_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/payments_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:37 /data/olist_proc/product_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:19 /data/olist_proc/products_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/reviews_df_cleaned.parquet
drwxr-xr

In [66]:
spark.stop()